In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np


def load_and_split(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values

def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values


In [ ]:
from catboost import CatBoostRegressor
import csv
import os

import sys
sys.path.append('..')

from src.model_generator import CopyModelGenerator, OptunaModelGenerator
from src.gradient_boosting_regressor import MyCatBoost
import optuna

RESULTS_PATH = "benchmark_results_2.csv"

def log_result_csv(
    model_name,
    name,
    r2_val,
    r2_test,
    n_trees,
    path=RESULTS_PATH
):
    file_exists = os.path.isfile(path)

    with open(path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "model",
                "dataset",
                "r2_eval",
                "r2_test",
                "n_trees"
            ])
        writer.writerow([
            model_name,
            name,
            f"{r2_val:.6f}",
            f"{r2_test:.6f}",
            n_trees
        ])

def test_random_boosting(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value
    print(X_train.shape)
    study = optuna.create_study(
        direction="minimize",  # RMSE
        sampler=optuna.samplers.TPESampler()
    )


    single_tree_model = CatBoostRegressor(
        iterations=1,
        learning_rate=1.0,
        loss_function='RMSE',
        verbose=False
    )
    model_generator = OptunaModelGenerator(single_tree_model, study)
    gbrt = MyCatBoost(
        model_generator=model_generator,
        n_estimators=2000,
        learning_rate=0.1,
        verbose=False
    )

    gbrt.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    log_result_csv(
        "random_boosting (proposal)",
        name,
        r2_val,
        r2,
        len(gbrt.models)
    )

def test_catboost_style_mine(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    single_tree_model = CatBoostRegressor(
        iterations=1,
        learning_rate=1.0,
        loss_function='RMSE',
        verbose=False
    )
    model_generator = CopyModelGenerator(single_tree_model)
    gbrt = MyCatBoost(
        model_generator=model_generator,
        n_estimators=2000,
        learning_rate=0.1,
        verbose=False
    )

    gbrt.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    log_result_csv(
        "catboost-style (self-made, default catboost HP)",
        name,
        r2_val,
        r2,
        len(gbrt.models)
    )

def test_catboost(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    gbrt = CatBoostRegressor(
        n_estimators=2000,
        learning_rate=0.1,
        loss_function='RMSE',
        verbose=False
    )

    gbrt.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    r2_val = r2_score(y_val, gbrt.predict(X_val))
    log_result_csv(
        "catboost (default HP)",
        name,
        r2_val,
        r2,
        gbrt.get_best_iteration() + 1
    )

def test_all_3(name, value):
    test_random_boosting(name, value)
    test_catboost_style_mine(name, value)
    test_catboost(name, value)

In [ ]:
# 1. California Housing (~20k)
cal = fetch_california_housing(as_frame=True)
test_all_3(
    "california_housing",
    load_and_split(cal.data.values, cal.target.values),
)


# 2. Bike Sharing Demand (~17k)
X, y = fetch_openml_numeric("Bike_Sharing_Demand", target="count")
test_all_3(
    "bike_sharing",
    load_and_split(X, y),
)


# 3. Medical Charges (~13k)
X, y = fetch_openml_numeric("medical_charges", target="charges")
test_all_3(
    "medical_charges",
    load_and_split(X, y),
)


# King County House Prices
X, y = fetch_openml_numeric_by_id(42165)
test_all_3(
    "king_county_house_prices",
    load_and_split(X, y),
)


# 1. Online News Popularity (shares)
# ~39k samples, noisy, heavy-tailed target
X, y = fetch_openml_numeric_by_id(42705)
test_all_3(
    "online_news_popularity",
    load_and_split(X, y),
)


# 2. YearPredictionMSD
# ~515k samples, но low-dim, можно сабсемплить
X, y = fetch_openml_numeric_by_id(44027)
test_all_3(
    "year_prediction_msd",
    load_and_split(
        X[:20_000],
        y[:20_000],  # безопасный сабсет
    ),
)


# 3. CPU Activity
# ~20k samples, классический UCI-style regression
X, y = fetch_openml_numeric_by_id(44963)
test_all_3(
    "cpu_activity",
    load_and_split(X, y),
)


# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
test_all_3(
    "facebook_comment_volume",
    load_and_split(X, y),
)


# 5. Airline Delay (departure delay)
# ~54k samples после очистки
X, y = fetch_openml_numeric_by_id(1169)
mask = np.isfinite(y)
test_all_3(
    "airline_delay",
    load_and_split(X[mask], y[mask]),
)


# 6. Superconductivity
# ~21k samples, физика, сложные взаимодействия
X, y = fetch_openml_numeric_by_id(44964)
test_all_3(
    "superconductivity",
    load_and_split(X, y),
)


# 7. Diamonds (price)
# ~54k samples
X, y = fetch_openml_numeric_by_id(42225)
test_all_3(
    "diamonds",
    load_and_split(X, y),
)


# 8. House Prices (Ames, extended)
# ~29k samples
X, y = fetch_openml_numeric_by_id(42563)
test_all_3(
    "ames_housing_large",
    load_and_split(X, y),
)


# 9. Brazilian Houses
# ~10k samples
X, y = fetch_openml_numeric_by_id(45020)
test_all_3(
    "brazilian_houses",
    load_and_split(X, y),
)


# 10. Metro Interstate Traffic Volume
# ~48k samples, сильная сезонность
X, y = fetch_openml_numeric_by_id(42477)
test_all_3(
    "metro_traffic_volume",
    load_and_split(X, y),
)


[I 2026-01-18 18:16:34,768] A new study created in memory with name: no-name-d93e2b6f-1410-48d6-b8f7-9240ed897ed1


(13209, 8)
Pre-algo time: 0.00037217140197753906 s
Times: Get Model: 22.828097343444824s    Fit: 25.108862161636353s    Pred: 14.476048707962036s    Other: 4.350008249282837s
Total Estim: 66.76301646232605s    Total: 66.9080605506897s
